In [7]:
import numpy as np
import pandas as pd

import os
import seaborn as sns

import matplotlib.pyplot as plt
from scipy import stats

import warnings
import json

from sklearn import manifold

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import OrdinalEncoder

import xgboost as xgb

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import confusion_matrix
from xgboost import plot_tree

warnings.filterwarnings('ignore')   

In [22]:
df = pd.read_csv("./train.csv")
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [36]:
def process_df(df):
    df = df.drop(['Name', 'PassengerId'], axis=1)
    df = df.dropna()

    y = df['Transported']
    df = df.drop(['Transported'], axis=1)
    y = y.astype(int)

    cb1 = df['Cabin'].str[0]
    cb2 = df['Cabin'].str[2]
    cb3 = df['Cabin'].str[4]

    df = df.drop(['Cabin'],axis=1)

    num_columns = list(df.select_dtypes(include=np.number).columns)
    cat_columns = list(df.select_dtypes(include=['object']).columns)

    df_num_col = df[num_columns]
    df_cat_col = df[cat_columns]

    encoder = OrdinalEncoder()
    df_cat_col = encoder.fit_transform(df_cat_col)

    # print(df_cat_col)

    df.drop(cat_columns, axis=1, inplace=True)

    i = 1
    for col in cat_columns:
        df["cat_" + str(i)] = df_cat_col[:, i-1] 
        i += 1

    df['Cabin1'] = cb1
    df['Cabin2'] = cb2
    df['Cabin3'] = cb3

    for col in num_columns:
        df[col] = df[col].fillna(df[col].mean())

    return df


In [37]:
process_df(df)

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,cat_1,cat_2,cat_3,cat_4,Cabin1,Cabin2,Cabin3
0,39.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,B,0,P
1,24.0,109.0,9.0,25.0,549.0,44.0,0.0,0.0,2.0,0.0,F,0,S
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1.0,0.0,2.0,1.0,A,0,S
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1.0,0.0,2.0,0.0,A,0,S
4,16.0,303.0,70.0,151.0,565.0,2.0,0.0,0.0,2.0,0.0,F,1,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8688,41.0,0.0,6819.0,0.0,1643.0,74.0,1.0,0.0,0.0,1.0,A,9,/
8689,18.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,G,1,9
8690,26.0,0.0,0.0,1872.0,1.0,0.0,0.0,0.0,2.0,0.0,G,1,0
8691,32.0,0.0,1049.0,0.0,353.0,3235.0,1.0,0.0,0.0,0.0,E,6,8


In [8]:
df.select_dtypes(include=np.number).columns

Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck'], dtype='object')